# model.py

In [145]:
# %% Dependencies
import torch 
import torch.distributions as td
import torch.nn as nn 
from torch_geometric.utils import dense_to_sparse

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import get_edge_type

# %%
class ContNodeFeats(nn.Module):
    def __init__(self, node_size, cont_node_feat):
        super(ContNodeFeats, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.cont_node_feat_1 = nn.Linear(self.node_size, 
            self.cont_node_feat * self.node_size)

    def forward(self):
        Z = torch.randn(self.node_size)
        X = self.cont_node_feat_1(Z)
        X = X.view(-1, self.cont_node_feat)
        return X

# %%
class DisNodeFeat(nn.Module): 
    def __init__(self, node_size, cell_types):
        super(DisNodeFeat, self).__init__()
        self.node_size = node_size
        self.cell_types = cell_types

        self.cell_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((1, self.cell_types))
            )
        )

    def forward(self):
        self.dist = td.Categorical(logits=self.cell_logits)
        self.sample = self.dist.sample([self.node_size]).squeeze()
        self.logLik = self.dist.log_prob(self.sample).sum()
        return self.sample + 1 # category indexing starts at 1. 

# %%
class AdjacencyMatrix(nn.Module):
    def __init__(self, node_size):
        super(AdjacencyMatrix, self).__init__()
        self.node_size = node_size

        self.edge_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((self.node_size, self.node_size))
            )
        )

    def forward(self):
        self.dist = td.Bernoulli(logits=self.edge_logits)
        self.sample = self.dist.sample().squeeze()
        self.logLik = self.dist.log_prob(self.sample).sum()
        return self.sample

# %% 
class EdgeFeats(nn.Module):
    def __init__(self, node_size, cont_edge_feat):
        super(EdgeFeats, self).__init__() 
        self.node_size = node_size
        self.cont_edge_feat = cont_edge_feat

    def forward(self, A, C_x): 
        # Discrete Edge Features:
        edges = list(zip(A.long()[0], A.long()[1]))
        e_c = list(map(lambda x: get_edge_type(x, C_x.int()), edges))
        e_c = torch.tensor(e_c)
        e_c = e_c.reshape(-1, 1)

        # Continuous Edge Features
        Z = torch.randn(self.node_size**2, 1)
        W = torch.normal(mean=0, std=1, size=(1, self.cont_edge_feat))
        E = Z @ W
        E = E[:A.shape[1]] # match number of edges.

        # Combines continuous and discrete node features.
        edge_features = torch.cat((e_c, E), dim=-1) 
        return edge_features


# %% Full Model 
class EGG(nn.Module):
    def __init__(self, node_size, cont_node_feat, cell_types, cont_edge_feat,): 
        super(EGG, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.cell_types = cell_types
        self.cont_edge_feat = cont_edge_feat

        # Sub-Generator models.
        self.ContNodeFeats = ContNodeFeats(self.node_size, self.cont_node_feat)
        self.DisNodeFeat = DisNodeFeat(self.node_size, self.cell_types)
        self.AdjacencyMatrix = AdjacencyMatrix(self.node_size)
        self.EdgeFeats = EdgeFeats(self.node_size, self.cont_edge_feat)

    def forward(self):
        # Sub-Generator models.
        X = self.ContNodeFeats()

        C_x = self.DisNodeFeat()

        A = self.AdjacencyMatrix()
        A = dense_to_sparse(A)[0]

        E = self.EdgeFeats(A, C_x)
        
        return X, C_x, A, E

# Edit-Distance.py

In [146]:
# %% Edit Distance
from typing import List

import networkx as nx
from networkx import graph_edit_distance

import torch_geometric

import numpy as np 
import multiprocessing as mp 
from multiprocessing import Pool
from functools import partial

sys.path.append("../scripts/ceograph/")
from ceograph import NucleiNet, NucleiData

mp.set_start_method('fork', force=True)

# Helper function for converting Nuclei Data to NetworkX while retaining features.
def nuclei_to_nx(data: NucleiData) -> nx.DiGraph: 

    G = nx.DiGraph()   

    # Add nodes with features
    for i in range(data.num_nodes):
        node_feats = np.hstack((data.cell_type[i].numpy(), data.x[i].numpy()))
        G.add_node(i, node_features=node_feats)

    # Add edges with features
    for i in range(data.num_edges):
        src, tgt = data.edge_index[0, i].item(), data.edge_index[1, i].item()
        G.add_edge(src, tgt, edge_features=data.edge_attr[i].numpy())

    return G


def node_strict_type_match(node_dict_1, node_dict_2): 

    # Quick return false if features names don't match. 
    if not(set(node_dict_1) & set(node_dict_2)):
        return 0

    if node_dict_1['node_features'][0] == node_dict_2['node_features'][0]:
        return 1

    else: 
        return 0 

def single_edit_distance(ob: nx.DiGraph, G: nx.DiGraph,
                         node_match=node_strict_type_match):
    dist = graph_edit_distance(
        ob, G, 
        node_match=node_match, 
        node_del_cost=lambda x: 0, 
        edge_del_cost=lambda x: 0,
        upper_bound=50, 
        timeout=60
    )

    if dist is None: 
        return torch.tensor([50.0])
    else:
        return torch.tensor([dist])

def list_edit_distance(G: nx.DiGraph, obs: List[nx.DiGraph], 
                       dist_fn=single_edit_distance):
    with Pool() as pool: 
        distances = pool.map(partial(dist_fn, G=G), obs)
    
    return torch.stack(distances).mean()


# loss.py

# utils.py

In [147]:
# %% Dependencies
from typing import Optional
import copy 
from torch_geometric.utils import remove_isolated_nodes

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData

# %%
def clear_iso_nodes(example: NucleiData, 
                    num_nodes: Optional[int] = None) -> NucleiData: 
    edge_index, _, mask = (
        remove_isolated_nodes(example.edge_index, num_nodes=num_nodes)
    )
    example_masked = NucleiData(
        x = example.x[mask], 
        edge_index = edge_index, 
        cell_type = example.cell_type[mask], 
        edge_attr = example.edge_attr,
    )

    return example_masked

# train.py

In [148]:
# %% Dependencies
import random 
import time
import pickle

from torch.optim import Adam

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData, load_model

# Load ad obs 
path = "../data/slides/LUDA/ad_train_nx_100.pkl"

with open(path, 'rb') as f:
    ad_train_nx_100 = pickle.load(f)

# %% Training Config

# Reproducibility 
random.seed(0)
torch.manual_seed(0)
device = torch.device(0)

# Model to be explained. 
explainee = load_model(path = "../data/trained/epoch_263.pt",
device=device)

# Model Parameters:
max_nodes = 50
cont_node_feats = 11
cell_types = 5
cont_edge_feat = 2

EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat)

# Dataloader

In [149]:
from torch.utils.data import DataLoader

class EGG_Loader:
    def __init__(self, EGG_Model, explainess, target, edit_obs):
        self.EGG_Model = EGG_Model
        self.explainee = explainee
        self.target = target
        self.edit_obs = edit_obs

    def __len__(self):
        return 1

    def __getitem__(self, _):
        for _ in range(100):
            try:
                X, C_x, A, E = self.EGG_Model()
                example = NucleiData(X, C_x, A, E)
                example = clear_iso_nodes(example).to(torch.device(0))
                explainee_pred = torch.softmax(self.explainee(example), dim=0).cpu()

                break

            except Exception as e:
                print({e})

        with torch.no_grad():
            example_nx = nuclei_to_nx(example.detach().cpu())
            edit_dist = list_edit_distance(example_nx, self.edit_obs)

        logLik_DisNodeFeat = self.EGG_Model.DisNodeFeat.logLik
        logLik_AdjacencyMatrix = self.EGG_Model.AdjacencyMatrix.logLik

        return explainee_pred, edit_dist, logLik_DisNodeFeat, logLik_AdjacencyMatrix

In [150]:
# %% Dependencies
import random 
import time
import pickle

from torch.optim import Adam
import torch.multiprocessing as mp
from torch.utils.data import DataLoader

# Set the multiprocessing start method to 'spawn'
mp.set_start_method('spawn', force=True)

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData, load_model

# Load ad obs 
path = "../data/slides/LUDA/ad_train_nx_100.pkl"

with open(path, 'rb') as f:
    ad_train_nx_100 = pickle.load(f)

# %% Training Config

# Reproducibility 
random.seed(0)
torch.manual_seed(0)
device = torch.device(0)

# Model to be explained. 
explainee = load_model(path = "../data/trained/epoch_263.pt",
device=device)

# Model Parameters:
max_nodes = 50
cont_node_feats = 11
cell_types = 5
cont_edge_feat = 2

EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat)

# Training Parameters:
num_epochs = 5
batch_size = 10
num_workers = batch_size
learning_rate = 1e-4
optimizer = Adam(EGG_Model.parameters(), lr=learning_rate)
target = torch.tensor([1.0, 0.0])
edit_obs = ad_train_nx_100
criterion = nn.BCELoss()
lambda_1 = 1
lambda_2 = 1
lambda_3 = 1

# %% Training Loop 
def main(): 
    for epoch in range(num_epochs): 
        start = time.time()

        optimizer.zero_grad()

        losses = [] # stored for plotting. 

        dataset = EGG_Loader(EGG_Model, explainee, target, edit_obs)
        dataloader = DataLoader(dataset, batch_size=batch_size)

        for batch in dataloader:
            explainee_preds, edit_dists, logLik_DisNodeFeat, logLik_AdjacencyMatrix = batch

            # Compute losses using the batched results
            pred_loss = criterion(explainee_preds, target) * (1 + logLik_DisNodeFeat + logLik_AdjacencyMatrix)
            edit_loss = edit_dists * (logLik_DisNodeFeat + logLik_AdjacencyMatrix)
        
            edge_pen = torch.norm(EGG_Model.AdjacencyMatrix.edge_logits, p=1)

            loss = (lambda_1 * pred_loss / batch_size
                + lambda_2 * edit_loss / batch_size
                + lambda_3 * edge_pen)

            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        end = time.time()

        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.10f}" + 
            f"Time: {end-start:.2f} seconds")

In [151]:
if __name__ == '__main__':
    main()

Process SpawnPoolWorker-79:
Traceback (most recent call last):
  File "/work/DPDS/s224833/Dissertation/gnn-egg/env/egg-env-38/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/work/DPDS/s224833/Dissertation/gnn-egg/env/egg-env-38/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/work/DPDS/s224833/Dissertation/gnn-egg/env/egg-env-38/lib/python3.8/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/work/DPDS/s224833/Dissertation/gnn-egg/env/egg-env-38/lib/python3.8/multiprocessing/queues.py", line 358, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'single_edit_distance' on <module '__main__' (built-in)>
Process SpawnPoolWorker-81:
Traceback (most recent call last):
  File "/work/DPDS/s224833/Dissertation/gnn-egg/env/egg-env-38/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/work/DPDS/s22